# Kinetic Investigation

This notebook loads an Excel results file, extracts the elapsed time steps together with `c(nmm)`, `c(imino)`, and `c(product)`, and plots the concentration profiles.

In [ ]:
import os
import importlib.util
import subprocess
import sys
from scipy.optimize import curve_fit
from datetime import datetime, time, timedelta
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")

cache_dir = Path.cwd() / ".cache"
mpl_config_dir = Path.cwd() / ".mplconfig"
cache_dir.mkdir(exist_ok=True)
mpl_config_dir.mkdir(exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(cache_dir))
os.environ.setdefault("MPLCONFIGDIR", str(mpl_config_dir))


## Read-in Data

In [ ]:
excel_path = Path("conc-time-profile.xlsx")
name_input = excel_path.stem

#Barrier calculated from DFT to compare with experimental values
dG_dft = 64.3 # kJ/mol
################################################################

analysis_start = 0 # Start of range of time points to analyze (0-based index).
analysis_stop = None  # End of range of time points to analyze (exclusive, 0-based index). Use None to include all remaining points.

df = pd.read_excel(excel_path, sheet_name=0)
df.columns = [col.strip() if isinstance(col, str) else col for col in df.columns]

for column in df.columns:
    if isinstance(column, str) and column.startswith("Unnamed:"):
        df = df.rename(columns={column: "time_step"})
        break


def normalize_header(value):
    if not isinstance(value, str):
        return value
    normalized = value.strip().lower()
    for old, new in (("_", " "), ("(", " "), (")", " "), ("[", " "), ("]", " "), ("/", " ")):
        normalized = normalized.replace(old, new)
    return " ".join(normalized.split())


normalized_columns = {column: normalize_header(column) for column in df.columns}

time_column_aliases = {
    "elapsed time": {"elapsed time"},
    "time_step": {"time step"},
    "time": {"time"},
    "t_h": {"t h", "t in h"},
    "t_min": {"t min", "t in min"},
}
time_unit_hints = {
    "elapsed time": None,
    "time_step": None,
    "time": None,
    "t_h": "h",
    "t_min": "min",
}

time_column = None
time_unit_hint = None
for label, aliases in time_column_aliases.items():
    time_column = next((column for column, normalized in normalized_columns.items() if normalized in aliases), None)
    if time_column is not None:
        time_unit_hint = time_unit_hints[label]
        break

concentration_aliases = {
    "c(nmm)": {"c nmm", "2a mol l", "nmm mol l"},
    "c(imino)": {"c imino", "iminoester mol l"},
    "c(product)": {"c product", "product mol l"},
}
concentration_column_map = {}
for canonical_name, aliases in concentration_aliases.items():
    match = next((column for column, normalized in normalized_columns.items() if normalized in aliases), None)
    if match is not None:
        concentration_column_map[canonical_name] = match

concentration_columns = list(concentration_aliases)

if time_column is None:
    raise KeyError(
        "Missing expected time column. Tried "
        f"{tuple(sorted({alias for aliases in time_column_aliases.values() for alias in aliases}))}. "
        f"Available columns: {list(df.columns)}"
    )

missing_columns = [column for column in concentration_columns if column not in concentration_column_map]
if missing_columns:
    raise KeyError(f"Missing expected columns: {missing_columns}. Available columns: {list(df.columns)}")


def to_elapsed_minutes(series: pd.Series, unit_hint=None) -> pd.Series:
    if pd.api.types.is_timedelta64_dtype(series):
        return series.dt.total_seconds() / 60

    if pd.api.types.is_datetime64_any_dtype(series):
        return (series - series.iloc[0]).dt.total_seconds() / 60

    numeric_values = pd.to_numeric(series, errors="coerce")
    if numeric_values.notna().all():
        if unit_hint == "h":
            return numeric_values * 60
        if unit_hint == "min":
            return numeric_values
        if numeric_values.between(0, 1).all():
            return numeric_values * 24 * 60
        return numeric_values

    timedelta_values = pd.to_timedelta(series.astype(str), errors="coerce")
    if timedelta_values.notna().all():
        return timedelta_values.dt.total_seconds() / 60

    datetime_values = pd.to_datetime(series.astype(str), errors="coerce")
    if datetime_values.notna().all():
        return (datetime_values - datetime_values.iloc[0]).dt.total_seconds() / 60

    def convert(value):
        if pd.isna(value):
            return float("nan")
        if isinstance(value, timedelta):
            return value.total_seconds() / 60
        if isinstance(value, time):
            return value.hour * 60 + value.minute + value.second / 60 + value.microsecond / 60000000
        if isinstance(value, datetime):
            return value.timestamp() / 60
        if isinstance(value, str):
            parsed = pd.to_timedelta(value, errors="coerce")
            if pd.notna(parsed):
                return parsed.total_seconds() / 60
        numeric_value = float(value)
        if unit_hint == "h":
            return numeric_value * 60
        if unit_hint == "min":
            return numeric_value
        if 0 <= numeric_value <= 1:
            return numeric_value * 24 * 60
        return numeric_value

    return series.map(convert)


kinetics = df[[time_column, *concentration_column_map.values()]].copy()
kinetics = kinetics.rename(
    columns={
        time_column: "time_raw",
        **{source: target for target, source in concentration_column_map.items()},
    }
)
kinetics[concentration_columns] = kinetics[concentration_columns].apply(pd.to_numeric, errors="coerce")
kinetics["time_step_min"] = to_elapsed_minutes(kinetics["time_raw"], unit_hint=time_unit_hint)
kinetics["time_step_min"] = kinetics["time_step_min"] - kinetics["time_step_min"].iloc[0]
kinetics["time_step_h"] = kinetics["time_step_min"] / 60
kinetics = kinetics.dropna(subset=["time_step_min", *concentration_columns]).sort_values("time_step_min")
kinetics_all = kinetics.reset_index(drop=True).copy()
kinetics = kinetics_all.iloc[analysis_start:analysis_stop].copy()

if kinetics.empty:
    raise ValueError(
        f"No rows selected for analysis with analysis_start={analysis_start} and analysis_stop={analysis_stop}."
    )

print(
    f"Using {len(kinetics)} of {len(kinetics_all)} time points from {excel_path.name} "
    f"(rows {analysis_start}:{analysis_stop})."
)
kinetics = kinetics.reset_index(drop=True)
kinetics[["time_step_min", "time_step_h", *concentration_columns]]




## Plot Concentration-Time Curve

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))

for column, label in {
    "c(nmm)": "c nmm",
    "c(imino)": "c imino",
    "c(product)": "c product",
}.items():
    ax.plot(
        kinetics["time_step_min"],
        kinetics[column],
        marker="o",
        markersize=8,
        linewidth=0,
        label=label,
        color={"c nmm": "forestgreen", "c imino": "steelblue", "c product": "darkred"}[label],
    )

#ax.set_title("Kinetic Profile")
ax.set_xlabel("Elapsed time / min")
ax.set_ylabel("Concentration / M")
ax.grid(False)
ax.legend()
plt.tight_layout()

plt.savefig(f"conc-time-profile.png", dpi=300)
plt.savefig(f"conc-time-profile.pdf", dpi=300)

plt.show()


## Logarithmic Plot to identify 1st order rate law

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))

for column, label in {
    "c(nmm)": "c nmm",
    "c(imino)": "c imino",
}.items():
    ax.plot(
        kinetics["time_step_min"],
        np.log(kinetics[column]),
        marker="o",
        linewidth=2,
        label=label,
        color = {"c nmm": "forestgreen", "c imino": "steelblue"}[label],
    )

ax.set_title("Logarithmic Kinetic Profile")
ax.set_xlabel("Elapsed time / min")
ax.set_ylabel("Logarithmic Concentration")
ax.legend()
plt.tight_layout()
plt.show()

## 1st order fit

In [ ]:
## cat concentration with experimental errors #########
##standard kinetics
c_cat = 0.0005 / 2.4 
c_cat_max =  0.00025 / 2.4
c_cat_min = 0.00075 / 2.4
##########################################

c0_imino = kinetics["c(imino)"].iloc[0]
c0_nmm = kinetics["c(nmm)"].iloc[0]
print(f"c_cat (nominal): {c_cat:.5f} M")
print(f"c_cat range: {c_cat_min:.5f} to {c_cat_max:.5f} M")

def imino_fit (t, k):
    return np.exp(- c_cat * k * t) * c0_imino
popt, pcov = curve_fit(imino_fit, kinetics["time_step_min"], kinetics["c(imino)"])
k_fit = popt[0]
#print(f"Fitted rate constant imino: {k_fit:.4e} min^-1")

def nmm_fit (t, k):
    return c0_nmm * np.exp(-c_cat * k * t)
popt2, pcov2 = curve_fit(nmm_fit, kinetics["time_step_min"], kinetics["c(nmm)"])
k_fit_nmm = popt2[0]
print(f"Fitted rate constant nmm k: {k_fit_nmm:.4e} min^-1")

def c_product_from_imino(t, k):
    return c0_imino - imino_fit(t, k)

def c_product_from_nmm(t, k):
    return c0_nmm - nmm_fit(t, k)


def c_product_fit(t, p_max, k):
    return p_max * (1 - np.exp(-c_cat * k * t))


def c_product_from_nmm_with_yield(t, yield_max, k):
    return yield_max * c_product_from_nmm(t, k)

popt_yield_1st, pcov_yield_1st = curve_fit(
    lambda t, yield_max: c_product_from_nmm_with_yield(t, yield_max, k_fit_nmm),
    kinetics["time_step_min"],
    kinetics["c(product)"],
    p0=[min(1.0, kinetics["c(product)"].max() / c0_nmm)],
    bounds=([0.0], [1.0]),
    maxfev=20000,
)
yield_max_fit_1st = popt_yield_1st[0]
print(f"Fitted max product yield cap: {yield_max_fit_1st:.3f} ({yield_max_fit_1st * 100:.1f}%)")
print(f"Product plateau from yield cap: {yield_max_fit_1st * c0_nmm:.5f} M")

plt.figure(figsize=(3.3, 3))
t_fit = np.linspace(0, kinetics["time_step_min"].max(), 100)
#plt.plot(kinetics["time_step_min"], kinetics["c(imino)"], 'o', color='steelblue', label='Iminoester')
plt.plot(kinetics["time_step_min"], kinetics["c(nmm)"], 'x', color='forestgreen', label='NMM')
plt.plot(kinetics["time_step_min"], kinetics["c(product)"], 'd', color='darkred', label='Product')
#plt.plot(t_fit, imino_fit(t_fit, k_fit), '-', color='steelblue')
plt.plot(t_fit, nmm_fit(t_fit, k_fit_nmm), '-', color='forestgreen')
plt.plot(
    t_fit,
    c_product_from_nmm_with_yield(t_fit, yield_max_fit_1st, k_fit_nmm),
    '-',
    color="darkred",
)

#plt.title("Imino Concentration Fit")
plt.xlabel("Elapsed time / min", fontsize="small")
plt.ylabel("Concentration", fontsize="small")
plt.legend(loc="upper right", fontsize="small", frameon=True, edgecolor="black")
plt.xticks(fontsize="small")
plt.yticks(fontsize="small")
plt.grid(False)
plt.tight_layout()


plt.savefig(f"kinetic_fit_1st_order_{name_input}.png", dpi=300)
plt.savefig(f"kinetic_fit_1st_order_{name_input}.pdf", dpi=300)

plt.show()  



In [ ]:
# Calculate dG from k using the Eyring equation
R = 8.314  # J/(mol*K)
T = 298.15  # K
k_B = 1.380649e-23  # J/K
h = 6.62607015e-34  # J*s
def calculate_dG(k):
    return -R * T * np.log((k * h) / (k_B * T))

def rescale_k_for_c_cat(k_fit, c_cat_actual):
    return k_fit * c_cat / c_cat_actual

k_fit_nmm_s = k_fit_nmm/60 # Convert from min^-1 to s^-1

dG = calculate_dG(k_fit_nmm_s)/1000  # Convert to kJ/mol
k_fit_nmm_min = rescale_k_for_c_cat(k_fit_nmm_s, c_cat_max)
k_fit_nmm_max = rescale_k_for_c_cat(k_fit_nmm_s, c_cat_min)

dG_min = calculate_dG(k_fit_nmm_max)/1000
dG_max = calculate_dG(k_fit_nmm_min)/1000

print(f"k (nominal c_cat): {k_fit_nmm_s:.4e} s^-1")
print(f"k range from c_cat: {k_fit_nmm_min:.4e} to {k_fit_nmm_max:.4e} s^-1")
print(f"k = {k_fit_nmm_s:.4e} (+{k_fit_nmm_max - k_fit_nmm_s:.4e}/-{k_fit_nmm_s - k_fit_nmm_min:.4e}) s^-1")
print("\n")
print(f"Calculated dG: {dG:.1f} kJ/mol")
print(f"Calculated dG range from c_cat: {dG_min:.1f} to {dG_max:.1f} kJ/mol")
print(f"Calculated dG: {dG:.1f} (+{dG_max - dG:.1f}/-{dG - dG_min:.1f}) kJ/mol")
print(f"Calculated dG: {dG*0.239006:.2f} (+{dG_max*0.239006 - dG*0.239006:.2f}/-{dG*0.239006 - dG_min*0.239006:.2f}) kcal/mol")
print(f"Calculated dG range from c_cat: {dG_min*0.239006:.2f} to {dG_max*0.239006:.2f} kcal/mol")

print(f"DFT dG: {dG_dft:.1f} kJ/mol")
k_from_dft = (k_B * T / h) * np.exp(-dG_dft * 1000 / (R * T))
print(f"Calculated k from DFT dG: {k_from_dft:.4e} s^-1")

## 2nd order fit

In [ ]:
def imino_fit_2nd_order(t, k):
    return (c0_imino*(c0_imino-c0_nmm)*np.exp((c0_imino-c0_nmm)*c_cat*k*t))/(c0_imino*np.exp((c0_imino-c0_nmm)*c_cat*k*t) - c0_nmm)

def nmm_fit_2nd_order(t, k):
    return (c0_nmm*(c0_nmm-c0_imino)*np.exp((c0_nmm-c0_imino)*c_cat*k*t))/(c0_nmm*np.exp((c0_nmm-c0_imino)*c_cat*k*t) - c0_imino)

def c_product_from_nmm_2nd_order(t, k):
    return c0_nmm - nmm_fit_2nd_order(t, k)


def c_product_from_nmm_2nd_order_with_yield(t, yield_max, k):
    return yield_max * c_product_from_nmm_2nd_order(t, k)

popt_2nd, pcov_2nd = curve_fit(imino_fit_2nd_order, kinetics["time_step_min"], kinetics["c(imino)"])
k2_fit = popt_2nd[0]
print(f"Fitted rate constant k: {k2_fit:.4e} M^-1 min^-1")

popt_2nd_nmm, pcov_2nd_nmm = curve_fit(nmm_fit_2nd_order, kinetics["time_step_min"], kinetics["c(nmm)"])
k2_fit_nmm = popt_2nd_nmm[0]
print(f"Fitted rate constant k: {k2_fit_nmm:.4e} M^-1 min^-1")

popt_yield, pcov_yield = curve_fit(
    lambda t, yield_max: c_product_from_nmm_2nd_order_with_yield(t, yield_max, k2_fit_nmm),
    kinetics["time_step_min"],
    kinetics["c(product)"],
    p0=[min(1.0, kinetics["c(product)"].max() / c0_nmm)],
    bounds=([0.0], [1.0]),
    maxfev=20000,
)
yield_max_fit = popt_yield[0]
print(f"Fitted max product yield cap: {yield_max_fit:.3f} ({yield_max_fit * 100:.1f}%)")
print(f"Product plateau from yield cap: {yield_max_fit * c0_nmm:.5f} M")

plt.figure(figsize=(3.3, 3))
t_fit = np.linspace(0, kinetics["time_step_min"].max(), 100)
#plt.plot(kinetics["time_step_min"], kinetics["c(imino)"], 'o', color='steelblue', label='Iminoester')
plt.plot(kinetics["time_step_min"], kinetics["c(nmm)"], 'x', color='forestgreen', label='NMM')
plt.plot(kinetics["time_step_min"], kinetics["c(product)"], 'd', color='darkred', label='Product')
#plt.plot(t_fit, imino_fit_2nd_order(t_fit, k2_fit), '-', color='steelblue')
plt.plot(t_fit, nmm_fit_2nd_order(t_fit, k2_fit_nmm), '-', color='forestgreen')

plt.plot(
    t_fit,
    c_product_from_nmm_2nd_order_with_yield(t_fit, yield_max_fit, k2_fit_nmm),
    '-',
    color="darkred",
    #label="Product fit with yield cap",
)
'''
plt.plot(
    t_fit,
    c_product_from_nmm_2nd_order(t_fit, k2_fit_nmm),
    '--',
    color="darkred",
    alpha=0.45,
    label="Product ideal mass balance",
)
'''
plt.grid(False)
plt.xlabel("Elapsed time / min", fontsize="small")
plt.xticks(fontsize="small")
plt.yticks(fontsize="small")
plt.ylabel("Concentration (M)", fontsize="small")
plt.legend(loc="upper right", fontsize="small", frameon=True, edgecolor="black")

plt.tight_layout()

plt.savefig(f"kinetic_fit_2nd_order_{name_input}.png", dpi=300)
plt.savefig(f"kinetic_fit_2nd_order_{name_input}.pdf", dpi=300)

plt.show()  



## Goodness of Fit for NMM Profiles

The goodness of fit is calculated from the residuals between the measured NMM concentration and the simulated NMM concentration at each sampled time point:

$$
\mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(c_{\mathrm{NMM},\mathrm{obs}}(t_i)-c_{\mathrm{NMM},\mathrm{fit}}(t_i)\right)^2}
$$

Lower RMSE means the simulated concentration profile is closer to the measured NMM trace. Because the first-order and second-order NMM fits each use one fitted parameter (`k`), the residual metrics and AICc can be compared directly.


In [ ]:
def calculate_fit_metrics(model_name, y_observed, y_fitted, n_parameters):
    residuals = y_observed - y_fitted
    n_points = len(y_observed)

    sse = np.sum(residuals**2)
    rmse = np.sqrt(np.mean(residuals**2))
    mae = np.mean(np.abs(residuals))
    ss_total = np.sum((y_observed - np.mean(y_observed))**2)
    r_squared = 1 - sse / ss_total
    adjusted_r_squared = 1 - (1 - r_squared) * (n_points - 1) / (n_points - n_parameters - 1)

    # AICc is the small-sample corrected Akaike information criterion.
    # Lower AICc indicates the better residual/error tradeoff.
    aic = n_points * np.log(sse / n_points) + 2 * n_parameters
    aicc = aic + (2 * n_parameters * (n_parameters + 1)) / (n_points - n_parameters - 1)
    bic = n_points * np.log(sse / n_points) + n_parameters * np.log(n_points)

    return {
        "model": model_name,
        "n_points": n_points,
        "n_parameters": n_parameters,
        "SSE / M^2": sse,
        "RMSE / M": rmse,
        "MAE / M": mae,
        "R^2": r_squared,
        "adjusted R^2": adjusted_r_squared,
        "AICc": aicc,
        "BIC": bic,
        "max abs residual / M": np.max(np.abs(residuals)),
    }


nmm_observed = kinetics["c(nmm)"].to_numpy(dtype=float)
time_min = kinetics["time_step_min"].to_numpy(dtype=float)

nmm_first_order_fit = nmm_fit(time_min, k_fit_nmm)
nmm_second_order_fit = nmm_fit_2nd_order(time_min, k2_fit_nmm)

nmm_fit_metrics = pd.DataFrame(
    [
        calculate_fit_metrics("1st order NMM", nmm_observed, nmm_first_order_fit, n_parameters=1),
        calculate_fit_metrics("2nd order NMM", nmm_observed, nmm_second_order_fit, n_parameters=1),
    ]
)

rmse_ratio = (
    nmm_fit_metrics.loc[nmm_fit_metrics["model"] == "2nd order NMM", "RMSE / M"].iloc[0]
    / nmm_fit_metrics.loc[nmm_fit_metrics["model"] == "1st order NMM", "RMSE / M"].iloc[0]
)
delta_aicc = (
    nmm_fit_metrics.loc[nmm_fit_metrics["model"] == "2nd order NMM", "AICc"].iloc[0]
    - nmm_fit_metrics.loc[nmm_fit_metrics["model"] == "1st order NMM", "AICc"].iloc[0]
)

print(f"2nd order / 1st order RMSE ratio: {rmse_ratio:.3f}")
print(f"Delta AICc (2nd - 1st): {delta_aicc:.2f}")

nmm_fit_metrics


In [ ]:
# Calculate dG from k using the Eyring equation
R = 8.314  # J/(mol*K)
T = 298.15  # K
k_B = 1.380649e-23  # J/K
h = 6.62607015e-34  # J*s
def calculate_dG(k):
    return -R * T * np.log((k * h) / (k_B * T))

def rescale_k_for_c_cat(k_fit, c_cat_actual):
    return k_fit * c_cat / c_cat_actual

k2_fit_nmm_s = k2_fit_nmm/60 # Convert from M^-1 min^-1 to M^-1 s^-1

dG = calculate_dG(k2_fit_nmm_s)/1000  # Convert to kJ/mol
k2_fit_nmm_min = rescale_k_for_c_cat(k2_fit_nmm_s, c_cat_max)
k2_fit_nmm_max = rescale_k_for_c_cat(k2_fit_nmm_s, c_cat_min)
dG_min = calculate_dG(k2_fit_nmm_max)/1000
dG_max = calculate_dG(k2_fit_nmm_min)/1000

print(f"k (nominal c_cat): {k2_fit_nmm_s:.4e} M^-1 s^-1")
print(f"k range from c_cat: {k2_fit_nmm_min:.4e} to {k2_fit_nmm_max:.4e} M^-1 s^-1")
print(f"k = {k2_fit_nmm_s:.4e} (+{k2_fit_nmm_max - k2_fit_nmm_s:.4e}/-{k2_fit_nmm_s - k2_fit_nmm_min:.4e}) M^-1 s^-1")
print("\n")
print(f"Calculated dG: {dG:.1f} kJ/mol")
print(f"Calculated dG range from c_cat: {dG_min:.1f} to {dG_max:.1f} kJ/mol")
print(f"Calculated dG: {dG:.1f} (+{dG_max - dG:.1f}/-{dG - dG_min:.1f}) kJ/mol")
print(f"Calculated dG: {dG*0.239006:.2f} (+{dG_max*0.239006 - dG*0.239006:.2f}/-{dG*0.239006 - dG_min*0.239006:.2f}) kcal/mol")
print(f"Calculated dG range from c_cat: {dG_min*0.239006:.2f} to {dG_max*0.239006:.2f} kcal/mol")


print(f"DFT dG: {dG_dft:.1f} kJ/mol")
print(f"DFT dG: {dG_dft*0.239006:.2f} kcal/mol")
k_from_dft = (k_B * T / h) * np.exp(-dG_dft * 1000 / (R * T))
print(f"Calculated k from DFT dG: {k_from_dft:.4e} M^-1 s^-1")